# Notebook for topic modeling 

# 0. Imports

In [1]:
## load packages 
import pandas as pd
import re
import numpy as np

## nltk imports
#!pip install nltk # can install on terminal or by uncommenting this line
#import nltk; nltk.download('punkt'); nltk.download('stopwords')
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

## sklearn imports
from sklearn.feature_extraction.text import CountVectorizer

## lda
#!pip install gensim # can install by uncommenting this line
from gensim import corpora
import gensim

## visualizing LDA--likely need to install
#!pip install pyLDAvis # can install by uncommenting this line
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
pyLDAvis.enable_notebook()

## print mult things
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## random
import random
import string; punctlist = [char for char in string.punctuation] # list of english punctuation marks

# 0. Load data

In [2]:
ab = pd.read_csv("../public_data/airbnb_text.zip")
ab.head()

,id,name,name_upper,neighbourhood_group,price
0,2539,Clean & quiet apt home by the park,CLEAN & QUIET APT HOME BY THE PARK,Brooklyn,149
1,2595,Skylit Midtown Castle,SKYLIT MIDTOWN CASTLE,Manhattan,225
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,THE VILLAGE OF HARLEM....NEW YORK !,Manhattan,150
3,3831,Cozy Entire Floor of Brownstone,COZY ENTIRE FLOOR OF BROWNSTONE,Brooklyn,89
4,5022,Entire Apt: Spacious Studio/Loft by central park,ENTIRE APT: SPACIOUS STUDIO/LOFT BY CENTRAL PARK,Manhattan,80


# 1. Preprocess documents

In this case, each name/name_upper, or listing title, we're treating as a document

## 1.1 Load stopwords list and augment with our own custom ones

In [4]:
list_stopwords = stopwords.words("english")

custom_words_toadd = ['apartment', 'new york', 'nyc',
                      'bronx', 'brooklyn',
                     'manhattan', 'queens', 
                      'staten island']

list_stopwords_new = list_stopwords + custom_words_toadd


In [4]:
list_stopwords

['i',
 'me',
 'my',
 'myself',
 'we',
 'our',
 'ours',
 'ourselves',
 'you',
 "you're",
 "you've",
 "you'll",
 "you'd",
 'your',
 'yours',
 'yourself',
 'yourselves',
 'he',
 'him',
 'his',
 'himself',
 'she',
 "she's",
 'her',
 'hers',
 'herself',
 'it',
 "it's",
 'its',
 'itself',
 'they',
 'them',
 'their',
 'theirs',
 'themselves',
 'what',
 'which',
 'who',
 'whom',
 'this',
 'that',
 "that'll",
 'these',
 'those',
 'am',
 'is',
 'are',
 'was',
 'were',
 'be',
 'been',
 'being',
 'have',
 'has',
 'had',
 'having',
 'do',
 'does',
 'did',
 'doing',
 'a',
 'an',
 'the',
 'and',
 'but',
 'if',
 'or',
 'because',
 'as',
 'until',
 'while',
 'of',
 'at',
 'by',
 'for',
 'with',
 'about',
 'against',
 'between',
 'into',
 'through',
 'during',
 'before',
 'after',
 'above',
 'below',
 'to',
 'from',
 'up',
 'down',
 'in',
 'out',
 'on',
 'off',
 'over',
 'under',
 'again',
 'further',
 'then',
 'once',
 'here',
 'there',
 'when',
 'where',
 'why',
 'how',
 'all',
 'any',
 'both',
 'each

## 1.2 Remove stopwords from lowercase version of corpus


In [5]:
## convert to lowercase and a list
corpus_lower = ab.name.str.lower().to_list()
corpus_lower[0:5]

## use wordpunct tokenize and filter out with one
example_listing = corpus_lower[3]
nostop_listing = [word for word in wordpunct_tokenize(example_listing) 
                          if word not in list_stopwords_new]
nostop_listing

['clean & quiet apt home by the park',
 'skylit midtown castle',
 'the village of harlem....new york !',
 'cozy entire floor of brownstone',
 'entire apt: spacious studio/loft by central park']

['cozy', 'entire', 'floor', 'brownstone']

In [13]:
# example_listing = corpus_lower[3]
# example_listing

In [6]:
# for w in wordpunct_tokenize(example_listing) :
#     if w not in list_stopwords_new:
#         print(w)

[word for word in wordpunct_tokenize(example_listing) 
                          if word not in list_stopwords_new]

['cozy', 'entire', 'floor', 'brownstone']

## 1.3 stem and remove non-alpha

Other contexts we may want to leave digits in

In [7]:
## initialize stemmer
porter = PorterStemmer()

## apply to one by iterating
## over the tokens in the list
example_listing_preprocess = [porter.stem(token) 
                            for token in nostop_listing 
                            if token.isalpha() and 
                            len(token) > 2]

example_listing_preprocess

['cozi', 'entir', 'floor', 'brownston']

In [8]:
for t in nostop_listing:
    print(porter.stem(t))

cozi
entir
floor
brownston


In [9]:
example_listing
example_listing_preprocess

'cozy entire floor of brownstone'

['cozi', 'entir', 'floor', 'brownston']

## 1.4 Activity 1

The above example performed preprocessing on a single Airbnb listing. We want to generalize this preprocessing across all listings.

- Embed step two (remove stopwords) and step three (stem) into one or two functions that take in a raw string (eg the raw text of an Airbnb review) and return a preprocessed string 
- Apply the function iteratively to preprocess all the texts in `corpus_lower`. Output could either be a list where each list element is a string of a list (e.g., `cozy brownstone apt`), or a list of lists where each element is a tokenized string (e.g., `['cozy', 'brownstone', 'apt'])`

Output is flexible: it could be a list of lists containing tokenized/stemmed text or a list of strings.

In [10]:
# your code here to define the function(s)

def process_string(s):
    nostop_listing = [word for word in wordpunct_tokenize(s) 
                          if word not in list_stopwords_new]
    example_listing_preprocess = [porter.stem(token) 
                            for token in nostop_listing 
                            if token.isalpha() and 
                            len(token) > 2]
    return example_listing_preprocess

# your code here to apply the function


In [11]:
ab[~ab.name.isna() ]

,id,name,name_upper,neighbourhood_group,price
0,2539,Clean & quiet apt home by the park,CLEAN & QUIET APT HOME BY THE PARK,Brooklyn,149
1,2595,Skylit Midtown Castle,SKYLIT MIDTOWN CASTLE,Manhattan,225
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,THE VILLAGE OF HARLEM....NEW YORK !,Manhattan,150
3,3831,Cozy Entire Floor of Brownstone,COZY ENTIRE FLOOR OF BROWNSTONE,Brooklyn,89
4,5022,Entire Apt: Spacious Studio/Loft by central park,ENTIRE APT: SPACIOUS STUDIO/LOFT BY CENTRAL PARK,Manhattan,80
...,...,...,...,...,...
48890,36484665,Charming one bedroom - newly renovated rowhouse,CHARMING ONE BEDROOM - NEWLY RENOVATED ROWHOUSE,Brooklyn,70
48891,36485057,Affordable room in Bushwick/East Williamsburg,AFFORDABLE ROOM IN BUSHWICK/EAST WILLIAMSBURG,Brooklyn,40
48892,36485431,Sunny Studio at Historical Neighborhood,SUNNY STUDIO AT HISTORICAL NEIGHBORHOOD,Manhattan,115
48893,36485609,43rd St. Time Square-cozy single bed,43RD ST. TIME SQUARE-COZY SINGLE BED,Manhattan,55


In [12]:
def rm_stop(raw: str):
    nostop_listing = [word for word in wordpunct_tokenize(raw)
                          if word not in list_stopwords_new]
    return nostop_listing

def get_stem(raw: list):
    stems = [porter.stem(token) for token in raw
             if token.isalpha() and
             len(token) > 2]
    return stems

def rm_stop_and_stem(raw: str):
    if type(raw) is not str:
        return
    return get_stem(rm_stop(raw))

# your code here to apply the function
[rm_stop_and_stem(w) for w in corpus_lower]

[['clean', 'quiet', 'apt', 'home', 'park'],
 ['skylit', 'midtown', 'castl'],
 ['villag', 'harlem', 'new', 'york'],
 ['cozi', 'entir', 'floor', 'brownston'],
 ['entir', 'apt', 'spaciou', 'studio', 'loft', 'central', 'park'],
 ['larg', 'cozi', 'midtown', 'east'],
 ['blissartsspac'],
 ['larg', 'furnish', 'room', 'near', 'way'],
 ['cozi', 'clean', 'guest', 'room', 'famili', 'apt'],
 ['cute', 'cozi', 'lower', 'east', 'side', 'bdrm'],
 ['beauti', 'upper', 'west', 'side'],
 ['central', 'near', 'broadway'],
 ['love', 'room', 'garden', 'best', 'area', 'legal', 'rental'],
 ['wonder', 'guest', 'bedroom', 'singl'],
 ['west', 'villag', 'nest', 'superhost'],
 ['stop', 'studio'],
 ['perfect', 'parent', 'garden'],
 ['chelsea', 'perfect'],
 ['hip', 'histor', 'brownston', 'backyard'],
 ['huge', 'upper', 'east', 'cental', 'park'],
 ['sweet', 'spaciou', 'loft'],
 ['cbg', 'ctybgd', 'helpshaiti'],
 ['cbg', 'help', 'haiti', 'room'],
 ['cbg', 'help', 'haiti'],
 ['maison', 'de', 'bohemian'],
 ['sunni', 'bedroo

In [13]:
# your code here to define the function(s)
def cleantext(raw_text_lower):
    # remove stopwords.
    nostopls = [one_tok for one_tok in wordpunct_tokenize(raw_text_lower)
                          if one_tok not in list_stopwords_new]
    # stem.
    stemmed = [porter.stem(one_tok)
                            for one_tok in nostopls
                            if one_tok.isalpha() and
                            len(one_tok) > 2]
    return stemmed

# your code here to apply the function

noNAnames = ab.name[ ~ab.name.isna() ]
corpus_lower_noNA = noNAnames.str.lower().to_list()
cleaned = [cleantext(entry) for entry in corpus_lower_noNA]

In [15]:
%%time
ab["processed_text"] = ab.name.apply(rm_stop_and_stem)
ab

CPU times: user 1.53 s, sys: 13.5 ms, total: 1.55 s
Wall time: 1.55 s


,id,name,name_upper,neighbourhood_group,price,processed_text
0,2539,Clean & quiet apt home by the park,CLEAN & QUIET APT HOME BY THE PARK,Brooklyn,149,"[clean, quiet, apt, home, park]"
1,2595,Skylit Midtown Castle,SKYLIT MIDTOWN CASTLE,Manhattan,225,"[skylit, midtown, castl]"
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,THE VILLAGE OF HARLEM....NEW YORK !,Manhattan,150,"[the, villag, harlem, new, york]"
3,3831,Cozy Entire Floor of Brownstone,COZY ENTIRE FLOOR OF BROWNSTONE,Brooklyn,89,"[cozi, entir, floor, brownston]"
4,5022,Entire Apt: Spacious Studio/Loft by central park,ENTIRE APT: SPACIOUS STUDIO/LOFT BY CENTRAL PARK,Manhattan,80,"[entir, apt, spaciou, studio, loft, central, p..."
...,...,...,...,...,...,...
48890,36484665,Charming one bedroom - newly renovated rowhouse,CHARMING ONE BEDROOM - NEWLY RENOVATED ROWHOUSE,Brooklyn,70,"[charm, one, bedroom, newli, renov, rowhous]"
48891,36485057,Affordable room in Bushwick/East Williamsburg,AFFORDABLE ROOM IN BUSHWICK/EAST WILLIAMSBURG,Brooklyn,40,"[afford, room, bushwick, east, williamsburg]"
48892,36485431,Sunny Studio at Historical Neighborhood,SUNNY STUDIO AT HISTORICAL NEIGHBORHOOD,Manhattan,115,"[sunni, studio, histor, neighborhood]"
48893,36485609,43rd St. Time Square-cozy single bed,43RD ST. TIME SQUARE-COZY SINGLE BED,Manhattan,55,"[time, squar, cozi, singl, bed]"


# 2. Create a document-term matrix and do some basic diagnostics (more manual approach)

Here we'll create a DTM first using the raw documents; in the activity, you'll create one using the preprocessed docs
that you created in activity 1

## 2.1 Define the dtm function and select data to transform into a document-term matrix

In [16]:
## function provided
def create_dtm(list_of_strings, metadata):
    """ 
    Function to create dense document-term matrix (DTM) from a list of strings and provided metadata. 
    A sparse DTM is a list of term_index/doc_index tuples: if a given term occurs in a given doc at least once, 
        then this count is listed as a tuple; if not, that term/doc pair is omitted. 
    In a dense DTM, each row is one text (e.g., an Airbnb listing), each column is a term, and 
        each cell indicates the frequency of that word in that text. 
    
    Parameters:
        list_of_strings (Series): each row contains a preprocessed string (need not be tokenized)
        metadata (DataFrame): contains document-level covariates
    
    Returns:
        Dense DTM with metadata on left and then one column per word in lexicon
    """
    
    # initialize a sklearn tokenizer; this helps us tokenize the preprocessed string input
    vectorizer = CountVectorizer(lowercase = True) 
    dtm_sparse = vectorizer.fit_transform(list_of_strings)
    print('Sparse matrix form:\n', dtm_sparse[:3]) # take a look at sparse representation
    print()
    
    # switch the dataframe from the sparse representation to the normal dense representation (so we can treat it as regular dataframe)
    dtm_dense_named = pd.DataFrame(dtm_sparse.todense(), columns=vectorizer.get_feature_names_out ())
    print('Dense matrix form:\n', dtm_dense_named.head()) # take a look at dense representation
    dtm_dense_named_withid = pd.concat([metadata.reset_index(), dtm_dense_named], axis = 1) # add back document-level covariates

    return(dtm_dense_named_withid)

In [17]:

## filter out na's
## for shorter runtime, random sampling of 1000
## get metadata for those
## and also renaming price col since it's likely to be corpus word
ab_small = ab.loc[~ab.name.isnull(),
           ['id', 'neighbourhood_group', 'price', 'name']].copy().rename(columns = {'price':
            'price_rawdata'}).sample(n = 1000, random_state = 422)

ab_small['name_lower'] = ab_small['name'].str.lower()
ab_small.head()

,id,neighbourhood_group,price_rawdata,name,name_lower
23821,19227560,Queens,100,Super Cozy!,super cozy!
22905,18560625,Brooklyn,30,Beautiful Private Bedroom by Prospect Park,beautiful private bedroom by prospect park
20426,16289576,Manhattan,80,Best Location on the Upper West Side! - Part II,best location on the upper west side! - part ii
2018,893413,Manhattan,2500,Architecturally Stunning Former Synagogue!,architecturally stunning former synagogue!
18790,14882137,Queens,50,"Large, beautiful room near Bushwick","large, beautiful room near bushwick"


## 2.2 Execute the dtm function to create the document-term matrix

In [18]:
## example application on raw lowercase texts; 
dtm_nopre = create_dtm(list_of_strings= ab_small.name_lower,
                      metadata = ab_small[['id', 'neighbourhood_group', 'price_rawdata']])



Sparse matrix form:
 <Compressed Sparse Row sparse matrix of dtype 'int64'
	with 17 stored elements and shape (3, 970)>
  Coords	Values
  (0, 841)	1
  (0, 281)	1
  (1, 152)	1
  (1, 693)	1
  (1, 157)	1
  (1, 205)	1
  (1, 698)	1
  (1, 653)	1
  (2, 165)	1
  (2, 537)	1
  (2, 637)	1
  (2, 856)	1
  (2, 902)	1
  (2, 939)	1
  (2, 774)	1
  (2, 657)	1
  (2, 471)	1

Dense matrix form:
    001  10  10m  10min  10mins  1100  12mins  14  15  15min  ...  yoga  york  \
0    0   0    0      0       0     0       0   0   0      0  ...     0     0   
1    0   0    0      0       0     0       0   0   0      0  ...     0     0   
2    0   0    0      0       0     0       0   0   0      0  ...     0     0   
3    0   0    0      0       0     0       0   0   0      0  ...     0     0   
4    0   0    0      0       0     0       0   0   0      0  ...     0     0   

   you  your  yu  zen  ღღღsteps  法拉盛中心私人房間獨立衛浴  溫馨大套房  獨一無二的紐約閣樓  
0    0     0   0    0         0              0      0          0  
1    0 

In [19]:
## show first set of rows/cols
dtm_nopre.head()

## show arbitrary later cols in resulting data
dtm_nopre.shape
dtm_nopre.iloc[0:5, 480:500]

,index,id,neighbourhood_group,price_rawdata,001,10,10m,10min,10mins,1100,...,yoga,york,you,your,yu,zen,ღღღsteps,法拉盛中心私人房間獨立衛浴,溫馨大套房,獨一無二的紐約閣樓
0,23821,19227560,Queens,100,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,22905,18560625,Brooklyn,30,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,20426,16289576,Manhattan,80,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2018,893413,Manhattan,2500,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,18790,14882137,Queens,50,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


(1000, 974)

,inclusive,incredible,incredibly,indoor,inn,inq,insane,int,interior,international,interns,invincible,inviting,inwood,island,it,italy,its,jefferson,jewel
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 2.3 Use that matrix/column sums to get basic summary stats of top words

In [20]:
## summing each col
top_terms = dtm_nopre[dtm_nopre.columns[4:]].sum(axis = 0)

## sorting from most frequent to least frequent
top_terms.sort_values(ascending = False)

in           367
room         244
private      163
bedroom      152
apartment    130
            ... 
gay            1
gente          1
geodesic       1
george         1
獨一無二的紐約閣樓      1
Length: 970, dtype: int64

## 2.4 Activity 2: repeat the above but using the preprocessed text data

- Stick with the same random sample of 1000 `ab_small`
- Apply the preprocessing steps from activity 1 to create a new column in `ab_small` with the preprocessed text (if you got stuck on that, try just removing stopwords)
- Use the `create_dtm` function to create a document-term matrix from the preprocessed data
- Use colsums to summarize

In [49]:
## your code here 

ab_small["name_pp"] = ab_small.name_lower.apply(rm_stop_and_stem)

In [51]:
ab_small.name_pp = ab_small.name_pp.apply(lambda x: " ".join(x))
ab_small.name_pp

23821                             super cozi
22905    beauti privat bedroom prospect park
20426        best locat upper west side part
2018        architectur stun former synagogu
18790         larg beauti room near bushwick
                        ...                 
33473              new york multi unit build
12905                     privat room bright
19158          bluebird eleg near time squar
18451                spaciou room time squar
16875                  delux sweep citi view
Name: name_pp, Length: 1000, dtype: object

In [52]:
dtm_pre = create_dtm(list_of_strings= ab_small.name_pp,
                      metadata = ab_small[['id', 'neighbourhood_group', 'price_rawdata']])

dtm_pre

Sparse matrix form:
 <Compressed Sparse Row sparse matrix of dtype 'int64'
	with 13 stored elements and shape (3, 753)>
  Coords	Values
  (0, 646)	1
  (0, 170)	1
  (1, 60)	1
  (1, 516)	1
  (1, 65)	1
  (1, 519)	1
  (1, 480)	1
  (2, 70)	1
  (2, 388)	1
  (2, 697)	1
  (2, 729)	1
  (2, 584)	1
  (2, 482)	1

Dense matrix form:
    abcd  abod  access  acidot  acogedor  across  address  ador  aesthet  \
0     0     0       0       0         0       0        0     0        0   
1     0     0       0       0         0       0        0     0        0   
2     0     0       0       0         0       0        0     0        0   
3     0     0       0       0         0       0        0     0        0   
4     0     0       0       0         0       0        0     0        0   

   afford  ...  yard  year  yellow  yoga  york  zen  ღღღstep  法拉盛中心私人房間獨立衛浴  \
0       0  ...     0     0       0     0     0    0        0              0   
1       0  ...     0     0       0     0     0    0        0        

,index,id,neighbourhood_group,price_rawdata,abcd,abod,access,acidot,acogedor,across,...,yard,year,yellow,yoga,york,zen,ღღღstep,法拉盛中心私人房間獨立衛浴,溫馨大套房,獨一無二的紐約閣樓
0,23821,19227560,Queens,100,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,22905,18560625,Brooklyn,30,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,20426,16289576,Manhattan,80,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2018,893413,Manhattan,2500,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,18790,14882137,Queens,50,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,33473,26463879,Brooklyn,65,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
996,12905,9823085,Brooklyn,65,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
997,19158,15233387,Manhattan,300,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
998,18451,14520743,Manhattan,95,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# 3. Use gensim to more automatically preprocess/estimate a topic model

## 3.1 Creating the objects to feed the LDA modeling function

Different outputs described below: 
- Tokenized and preprocessed text 
- Dictionary 
- Corpus 

In [53]:

## Step 1: re-tokenize and store in list
## here, i'm doing with the raw random sample of text
## in activity, you should do with the preprocessed texts
text_raw_tokens = [wordpunct_tokenize(one_text) for one_text in 
                  ab_small.name_lower]


## Step 2: use gensim create dictionary - gets all unique words across documents
text_raw_dict = corpora.Dictionary(text_raw_tokens)
raw_len = len(text_raw_dict) # get length for comparison below

### explore first few keys and values
### see that key is just an arbitrary counter; value is the word itself
{k: text_raw_dict[k] for k in list(text_raw_dict)[:5]}


## Step 3: filter out very rare and very common words
## here, i'm using the threshold that a word needs to appear in at least
## 5% of docs but not more than 95%
## this is an integer count of docs so i round
lower_bound = round(ab_small.shape[0]*0.05)
upper_bound = round(ab_small.shape[0]*0.95)

### apply filtering to dictionary
text_raw_dict.filter_extremes(no_below = lower_bound,
                             no_above = upper_bound)
print(f'Filtering out very rare and very common words reduced the \
length of dictionary from {str(raw_len)} to {str(len(text_raw_dict))}.')
{k: text_raw_dict[k] for k in list(text_raw_dict)[:5]} # show first five entries after filtering


## Step 4: apply dictionary to TOKENIZED texts
## this creates a mapping between each word 
## in a specific listing and the key in the dictionary.
## for words that remain in the filtered dictionary,
## output is a list where len(list) == n documents
## and each element in the list is a list of tuples
## containing the mappings
corpus_fromdict = [text_raw_dict.doc2bow(one_text) 
                   for one_text in text_raw_tokens]

### can apply doc2bow(one_text, return_missing = True) to print words
### eliminated from the listing bc they're not in filtered dictionary.
### but feeding that one with missing values to
### the lda function can cause errors
corpus_fromdict_showmiss = [text_raw_dict.doc2bow(one_text, return_missing = True)
                            for one_text in text_raw_tokens]
print('Sample of documents represented in dictionary format (with omitted words noted):')
corpus_fromdict_showmiss[:10]

{0: '!', 1: 'cozy', 2: 'super', 3: 'beautiful', 4: 'bedroom'}

Filtering out very rare and very common words reduced the length of dictionary from 1047 to 31.


{0: '!', 1: 'cozy', 2: 'beautiful', 3: 'bedroom', 4: 'park'}

Sample of documents represented in dictionary format (with omitted words noted):


[([(0, 1), (1, 1)], {'super': 1}),
 ([(2, 1), (3, 1), (4, 1), (5, 1)], {'by': 1, 'prospect': 1}),
 ([(0, 1), (6, 1), (7, 1)],
  {'best': 1,
   'ii': 1,
   'location': 1,
   'on': 1,
   'part': 1,
   'side': 1,
   'upper': 1,
   'west': 1}),
 ([(0, 1)],
  {'architecturally': 1, 'former': 1, 'stunning': 1, 'synagogue': 1}),
 ([(2, 1), (8, 1), (9, 1), (10, 1), (11, 1)], {'bushwick': 1}),
 ([(4, 1), (8, 1), (9, 1), (12, 1), (13, 2)],
  {'bath': 1, 'bed': 1, 'by': 1, 'central': 1, 'college': 1, 'hunter': 1}),
 ([(9, 1), (11, 1), (14, 1), (15, 1)], {'bohemian': 1, 'brownstone': 1}),
 ([(16, 1)],
  {'fidi': 1, 'huge': 1, 'loft': 1, 'views': 1, 'w': 1, 'water': 1}),
 ([], {'hillside': 1, 'hotel': 1}),
 ([(5, 1), (9, 1), (11, 1), (14, 1), (15, 1)], {'airy': 1})]

##  3.2 Estimating the model

In [54]:
## Step 5: we're finally ready to estimate the model!
## full documentation here - https://radimrehurek.com/gensim/models/ldamodel.html
## here, we're feeding the lda function:
## (1) the corpus we created from the dictionary,
## (2) a parameter we decide on for the number of topics (k),
## (3) the dictionary itself,
## (4) parameter for number of passes through training data (more means slower), and
## (5) parameter that returns, for each word remaining in dict, the topic probabilities.
## see documentation for many other arguments you can vary
ldamod = gensim.models.ldamodel.LdaModel(corpus_fromdict, 
                                         num_topics = 5, 
                                         id2word=text_raw_dict, 
                                         passes=6, 
                                         alpha = 'auto',
                                         per_word_topics = True)

print(type(ldamod))



<class 'gensim.models.ldamodel.LdaModel'>


## 3.3  Seeing what topics the estimated model discovers

In [55]:
## Post-model 1: explore corpus-wide summary of topics
### getting the topics and top words; can retrieve diff top words
topics = ldamod.print_topics(num_words = 10)
for topic in topics:
    print(topic)


(0, '0.119*"studio" + 0.095*"." + 0.085*"1" + 0.080*"spacious" + 0.070*"apt" + 0.066*"in" + 0.058*"to" + 0.057*"bedroom" + 0.056*"-" + 0.055*","')
(1, '0.231*"in" + 0.171*"room" + 0.105*"private" + 0.095*"!" + 0.064*"the" + 0.061*"williamsburg" + 0.053*"of" + 0.031*"apt" + 0.028*"cozy" + 0.021*"sunny"')
(2, '0.098*"east" + 0.094*"/" + 0.088*"with" + 0.087*"&" + 0.086*"park" + 0.073*"bedroom" + 0.065*"beautiful" + 0.049*"-" + 0.043*"apartment" + 0.042*"in"')
(3, '0.127*"apartment" + 0.107*"bedroom" + 0.107*"in" + 0.102*"-" + 0.092*"2" + 0.088*"manhattan" + 0.051*"to" + 0.038*"," + 0.038*"private" + 0.037*"cozy"')
(4, '0.258*"," + 0.093*"and" + 0.082*"brooklyn" + 0.078*"room" + 0.066*"/" + 0.056*"cozy" + 0.045*"near" + 0.033*"!" + 0.033*"in" + 0.028*"to"')


In [56]:
    
## Post-model 2: explore topics associated with each document
### for each item in our original dictionary, get list of topic probabilities
l=[ldamod.get_document_topics(item) for item in corpus_fromdict]
### print result
text_raw_tokens[0:5]
l[0:5]

[['super', 'cozy', '!'],
 ['beautiful', 'private', 'bedroom', 'by', 'prospect', 'park'],
 ['best',
  'location',
  'on',
  'the',
  'upper',
  'west',
  'side',
  '!',
  '-',
  'part',
  'ii'],
 ['architecturally', 'stunning', 'former', 'synagogue', '!'],
 ['large', ',', 'beautiful', 'room', 'near', 'bushwick']]

[[(0, np.float32(0.049772836)),
  (1, np.float32(0.78855747)),
  (2, np.float32(0.04528242)),
  (3, np.float32(0.06334531)),
  (4, np.float32(0.053041983))],
 [(0, np.float32(0.028948788)),
  (1, np.float32(0.042724907)),
  (2, np.float32(0.8608049)),
  (3, np.float32(0.03685101)),
  (4, np.float32(0.03067042))],
 [(0, np.float32(0.036673285)),
  (1, np.float32(0.60122144)),
  (2, np.float32(0.033358105)),
  (3, np.float32(0.28994274)),
  (4, np.float32(0.038804468))],
 [(0, np.float32(0.07764929)),
  (1, np.float32(0.67159396)),
  (2, np.float32(0.070652224)),
  (3, np.float32(0.097771525)),
  (4, np.float32(0.082333))],
 [(0, np.float32(0.023956712)),
  (1, np.float32(0.035258017)),
  (2, np.float32(0.021858705)),
  (3, np.float32(0.030389892)),
  (4, np.float32(0.88853663))]]

### Visualizing 

In [57]:
lda_display = gensimvis.prepare(ldamod, corpus_fromdict, text_raw_dict)
pyLDAvis.display(lda_display)

## 3.4 Activity 3

- Preprocess the texts if you haven't already
- Run the topic model with preprocessed texts
- Play around with other parameters like `n_topics` to find a configuration that produces useful topics

If you get stuck on the preprocessing part, you can use below function and example code for applying it. Then continue as above (start with tokenizing).

In [58]:

## Step 1: re-tokenize and store in list
## here, i'm doing with the raw random sample of text
## in activity, you should do with the preprocessed texts
text_raw_tokens = [wordpunct_tokenize(one_text) for one_text in 
                  ab_small.name_pp]


## Step 2: use gensim create dictionary - gets all unique words across documents
text_raw_dict = corpora.Dictionary(text_raw_tokens)
raw_len = len(text_raw_dict) # get length for comparison below

### explore first few keys and values
### see that key is just an arbitrary counter; value is the word itself
{k: text_raw_dict[k] for k in list(text_raw_dict)[:5]}


## Step 3: filter out very rare and very common words
## here, i'm using the threshold that a word needs to appear in at least
## 5% of docs but not more than 95%
## this is an integer count of docs so i round
lower_bound = round(ab_small.shape[0]*0.05)
upper_bound = round(ab_small.shape[0]*0.95)

### apply filtering to dictionary
text_raw_dict.filter_extremes(no_below = lower_bound,
                             no_above = upper_bound)
print(f'Filtering out very rare and very common words reduced the \
length of dictionary from {str(raw_len)} to {str(len(text_raw_dict))}.')
{k: text_raw_dict[k] for k in list(text_raw_dict)[:5]} # show first five entries after filtering


## Step 4: apply dictionary to TOKENIZED texts
## this creates a mapping between each word 
## in a specific listing and the key in the dictionary.
## for words that remain in the filtered dictionary,
## output is a list where len(list) == n documents
## and each element in the list is a list of tuples
## containing the mappings
corpus_fromdict = [text_raw_dict.doc2bow(one_text) 
                   for one_text in text_raw_tokens]

### can apply doc2bow(one_text, return_missing = True) to print words
### eliminated from the listing bc they're not in filtered dictionary.
### but feeding that one with missing values to
### the lda function can cause errors
corpus_fromdict_showmiss = [text_raw_dict.doc2bow(one_text, return_missing = True)
                            for one_text in text_raw_tokens]
print('Sample of documents represented in dictionary format (with omitted words noted):')
corpus_fromdict_showmiss[:10]

{0: 'cozi', 1: 'super', 2: 'beauti', 3: 'bedroom', 4: 'park'}

Filtering out very rare and very common words reduced the length of dictionary from 753 to 14.


{0: 'cozi', 1: 'beauti', 2: 'bedroom', 3: 'park', 4: 'privat'}

Sample of documents represented in dictionary format (with omitted words noted):


[([(0, 1)], {'super': 1}),
 ([(1, 1), (2, 1), (3, 1), (4, 1)], {'prospect': 1}),
 ([], {'best': 1, 'locat': 1, 'part': 1, 'side': 1, 'upper': 1, 'west': 1}),
 ([], {'architectur': 1, 'former': 1, 'stun': 1, 'synagogu': 1}),
 ([(1, 1), (5, 1), (6, 1), (7, 1)], {'bushwick': 1}),
 ([(3, 1), (5, 1)],
  {'bath': 1, 'bed': 1, 'central': 1, 'colleg': 1, 'hunter': 1}),
 ([(5, 1), (7, 1)], {'bohemian': 1, 'brownston': 1}),
 ([(8, 1)], {'fidi': 1, 'huge': 1, 'loft': 1, 'view': 1, 'water': 1}),
 ([], {'hillsid': 1, 'hotel': 1}),
 ([(4, 1), (5, 1), (7, 1)], {'airi': 1})]

In [59]:
## Step 5: we're finally ready to estimate the model!
## full documentation here - https://radimrehurek.com/gensim/models/ldamodel.html
## here, we're feeding the lda function:
## (1) the corpus we created from the dictionary,
## (2) a parameter we decide on for the number of topics (k),
## (3) the dictionary itself,
## (4) parameter for number of passes through training data (more means slower), and
## (5) parameter that returns, for each word remaining in dict, the topic probabilities.
## see documentation for many other arguments you can vary
ldamod = gensim.models.ldamodel.LdaModel(corpus_fromdict, 
                                         num_topics = 5, 
                                         id2word=text_raw_dict, 
                                         passes=6, 
                                         alpha = 'auto',
                                         per_word_topics = True)

print(type(ldamod))



<class 'gensim.models.ldamodel.LdaModel'>


In [60]:
## Post-model 1: explore corpus-wide summary of topics
### getting the topics and top words; can retrieve diff top words
topics = ldamod.print_topics(num_words = 10)
for topic in topics:
    print(topic)


(0, '0.430*"privat" + 0.231*"room" + 0.163*"larg" + 0.103*"bedroom" + 0.034*"apt" + 0.020*"cozi" + 0.006*"spaciou" + 0.005*"east" + 0.004*"beauti" + 0.002*"williamsburg"')
(1, '0.343*"bedroom" + 0.257*"studio" + 0.144*"park" + 0.137*"spaciou" + 0.059*"east" + 0.016*"williamsburg" + 0.014*"cozi" + 0.011*"larg" + 0.005*"privat" + 0.005*"room"')
(2, '0.359*"sunni" + 0.285*"near" + 0.116*"park" + 0.056*"room" + 0.054*"spaciou" + 0.047*"privat" + 0.045*"bedroom" + 0.011*"larg" + 0.010*"beauti" + 0.009*"apt"')
(3, '0.423*"cozi" + 0.365*"apt" + 0.150*"east" + 0.019*"beauti" + 0.015*"spaciou" + 0.011*"studio" + 0.008*"williamsburg" + 0.002*"bedroom" + 0.002*"sunni" + 0.001*"room"')
(4, '0.489*"room" + 0.195*"williamsburg" + 0.173*"beauti" + 0.092*"spaciou" + 0.014*"park" + 0.014*"apt" + 0.005*"bedroom" + 0.005*"near" + 0.004*"cozi" + 0.003*"sunni"')


In [61]:
lda_display = gensimvis.prepare(ldamod, corpus_fromdict, text_raw_dict)
pyLDAvis.display(lda_display)